# RetainIQ — Phase 7.4: Executive Memo & MySQL Publishing

## Objective

I turn the Phase 7 outputs into the business-facing deliverables:

- a one-page executive memo
- a self-contained business summary for future RAG use
- MySQL reporting tables and views
- final CSV artifacts

The memo follows this exact structure:

**Problem → Key Finding → Recommendation → Expected Impact → Limitations**

In [1]:
from pathlib import Path
import json
import pandas as pd
import mysql.connector
from mysql.connector import Error
from IPython.display import display

ROOT = Path("..")
OUTPUT_DIR = ROOT / "outputs"
CONFIG_DIR = ROOT / "config"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def print_result(message):
    print(f"Result: {message}")


In [2]:
if "print_result" not in globals():
    def print_result(message):
        print(f"Result: {message}")

strategy_brief = pd.read_csv(OUTPUT_DIR / "phase_07_strategy_brief.csv").iloc[0]
segment_priority = pd.read_csv(OUTPUT_DIR / "segment_priority.csv")
state_priority = pd.read_csv(OUTPUT_DIR / "state_priority.csv")
city_priority = pd.read_csv(OUTPUT_DIR / "city_priority.csv")
action_framework = pd.read_csv(OUTPUT_DIR / "retention_action_framework.csv")
impact_scenarios = pd.read_csv(OUTPUT_DIR / "retention_impact_scenarios.csv")

print_result = lambda message: print(f"Result: {message}")

print_result(
    f"Loaded the executive-memo inputs: {len(segment_priority)} segments, {len(state_priority)} states, {len(city_priority):,} cities, and {len(action_framework)} action-framework rows."
)

Result: Loaded the executive-memo inputs: 3 segments, 1 states, 1,106 cities, and 3 action-framework rows.


### Result & conclusion

The final notebook is using the previously validated and prioritized outputs. I am not recomputing churn or revenue exposure here; this notebook is focused on communication and publication.

In [3]:
top_segment = segment_priority.iloc[0]
top_city = city_priority.iloc[0]

# Match action language to the top segment's observed retention profile.
action_row = action_framework[action_framework["segment_name"] == top_segment["segment_name"]].iloc[0]

print_result(
    f"The memo inputs center on {top_segment['segment_name']} ({top_segment['retention_profile']}) and {top_city['state']} — {top_city['city']}."
)
display(pd.DataFrame([{
    "priority_segment": top_segment["segment_name"],
    "segment_profile": top_segment["retention_profile"],
    "segment_revenue_at_risk": top_segment["revenue_at_risk"],
    "segment_churn_rate_pct": top_segment["churn_rate_pct"],
    "city": f"{top_city['state']} — {top_city['city']}",
    "city_revenue_at_risk": top_city["revenue_at_risk"],
    "city_churn_rate_pct": top_city["churn_rate_pct"],
    "action_theme": action_row["action_theme"],
}]))

Result: The memo inputs center on Segment 0 — Emerging Risk (Retention Priority) and California — San Diego.


,priority_segment,segment_profile,segment_revenue_at_risk,segment_churn_rate_pct,city,city_revenue_at_risk,city_churn_rate_pct,action_theme
0,Segment 0 — Emerging Risk,Retention Priority,1944581.19,46.99,California — San Diego,385446.39,64.91,Proactive churn-prevention intervention


### Result & conclusion

The executive narrative is anchored to one segment-level priority and one geographic exposure point. I will keep the recommendation operational and tie it directly to the historical revenue-at-risk figures and the scenario model.

In [4]:
recovery_rate = float(strategy_brief["scenario_recovery_rate_pct"]) / 100
segment_scenario = float(top_segment["revenue_at_risk"]) * recovery_rate
city_scenario = float(top_city["revenue_at_risk"]) * recovery_rate

print_result(
    f"At the configured {recovery_rate:.0%} recovery scenario, the top segment corresponds to ${segment_scenario:,.2f} and the top city corresponds to ${city_scenario:,.2f} of scenario recovery."
)

Result: At the configured 10% recovery scenario, the top segment corresponds to $194,458.12 and the top city corresponds to $38,544.64 of scenario recovery.


### Result & conclusion

The expected-impact language will be framed as a **scenario estimate**. I will not present it as a validated forecast because the recovery rate is an assumption rather than a model calibrated on intervention outcomes.

In [5]:
# Build the exact executive memo structure requested by the Phase 7 roadmap.
problem = (
    f"RetainIQ shows material historical revenue exposure among customers who have already churned, "
    f"with segment and market patterns that can be translated into focused retention activity."
)

finding_1 = (
    f"{top_segment['segment_name']} is the highest weighted segment priority, with "
    f"${top_segment['revenue_at_risk']:,.2f} of historical revenue at risk and a recorded churn rate of "
    f"{top_segment['churn_rate_pct']:.1f}%."
)
finding_2 = (
    f"{top_city['state']} — {top_city['city']} has the largest city-level historical revenue exposure in the current Phase 7 prioritization, "
    f"at ${top_city['revenue_at_risk']:,.2f}, with a churn rate of {top_city['churn_rate_pct']:.1f}%."
)
finding_3 = (
    f"A {recovery_rate:.0%} recovery scenario corresponds to ${segment_scenario:,.2f} for the top segment and "
    f"${city_scenario:,.2f} for the top city, illustrating the dollar scale of the exposure."
)

recommendation = (
    f"Prioritize {action_row['action_theme'].lower()} for {top_segment['segment_name']} and focus operational attention on "
    f"{top_city['state']} — {top_city['city']}. The decision is anchored to ${top_segment['revenue_at_risk']:,.2f} of segment exposure "
    f"and ${top_city['revenue_at_risk']:,.2f} of city-level exposure."
)

expected_impact = (
    f"Under the explicit {recovery_rate:.0%} recovery scenario, the top segment represents approximately ${segment_scenario:,.2f} of recoverable historical exposure."
)

limitations = [
    "Revenue at risk is a historical exposure proxy based on revenue from customers already recorded as churned; it is not a forecast of future loss.",
    "Ease-of-intervention scores are subjective business inputs, and the priority score changes when those assumptions change.",
    "Geographic patterns are descriptive and do not establish that location causes churn or that an intervention will produce the modeled recovery rate."
]

memo = f"""# RetainIQ — Executive Memo

## Problem
{problem}

## Key Finding
- {finding_1}
- {finding_2}
- {finding_3}

## Recommendation
{recommendation}

## Expected Impact
{expected_impact}

## Limitations
- {limitations[0]}
- {limitations[1]}
- {limitations[2]}
"""

memo_path = OUTPUT_DIR / "phase_07_executive_memo.md"
memo_path.write_text(memo, encoding="utf-8")

print_result(f"Generated the one-page executive memo at {memo_path}.")
print(memo)

Result: Generated the one-page executive memo at ..\outputs\phase_07_executive_memo.md.
# RetainIQ — Executive Memo

## Problem
RetainIQ shows material historical revenue exposure among customers who have already churned, with segment and market patterns that can be translated into focused retention activity.

## Key Finding
- Segment 0 — Emerging Risk is the highest weighted segment priority, with $1,944,581.19 of historical revenue at risk and a recorded churn rate of 47.0%.
- California — San Diego has the largest city-level historical revenue exposure in the current Phase 7 prioritization, at $385,446.39, with a churn rate of 64.9%.
- A 10% recovery scenario corresponds to $194,458.12 for the top segment and $38,544.64 for the top city, illustrating the dollar scale of the exposure.

## Recommendation
Prioritize proactive churn-prevention intervention for Segment 0 — Emerging Risk and focus operational attention on California — San Diego. The decision is anchored to $1,944,581.19 o

### Result & conclusion

The executive memo now follows the requested structure exactly. The recommendation is tied to observed revenue exposure, while the impact statement is explicitly tied to the configurable recovery scenario rather than presented as a guaranteed result.

In [6]:
business_summary = {
    "phase": "Phase 7",
    "objective": "Retention strategy and business intelligence layer",
    "top_segment": {
        "segment_name": str(top_segment["segment_name"]),
        "retention_profile": str(top_segment["retention_profile"]),
        "customers": int(top_segment["customers"]),
        "churn_rate_pct": float(top_segment["churn_rate_pct"]),
        "revenue_at_risk": float(top_segment["revenue_at_risk"]),
        "ease_of_intervention_score": int(top_segment["ease_of_intervention_score"]),
        "priority_score": float(top_segment["priority_score"])
    },
    "top_city": {
        "state": str(top_city["state"]),
        "city": str(top_city["city"]),
        "customers": int(top_city["customers"]),
        "churn_rate_pct": float(top_city["churn_rate_pct"]),
        "revenue_at_risk": float(top_city["revenue_at_risk"]),
        "priority_score": float(top_city["priority_score"]),
        "priority_basis": str(top_city["priority_basis"])
    },
    "scenario": {
        "recovery_rate_pct": float(strategy_brief["scenario_recovery_rate_pct"]),
        "top_segment_scenario_recovery": float(segment_scenario),
        "top_city_scenario_recovery": float(city_scenario)
    },
    "memo_path": str(memo_path)
}

json_path = OUTPUT_DIR / "phase_07_business_summary.json"
json_path.write_text(json.dumps(business_summary, indent=2), encoding="utf-8")
print_result(f"Saved a self-contained business summary for future RAG/GenAI use at {json_path}.")
display(pd.DataFrame([business_summary["top_segment"], business_summary["top_city"]], index=["segment", "city"]))

Result: Saved a self-contained business summary for future RAG/GenAI use at ..\outputs\phase_07_business_summary.json.


,segment_name,retention_profile,customers,churn_rate_pct,revenue_at_risk,ease_of_intervention_score,priority_score,state,city,priority_basis
segment,Segment 0 — Emerging Risk,Retention Priority,3226,46.99,1944581.19,4.0,7778324.76,NaN,NaN,NaN
city,NaN,NaN,285,64.91,385446.39,NaN,385446.39,California,San Diego,Revenue at Risk only


### Result & conclusion

The Phase 7 narrative is also available as structured JSON. This keeps the business findings easy to ingest later into the GenAI/RAG layer without asking the model to reconstruct the analysis from a notebook.

In [9]:
from getpass import getpass
MYSQL_CONFIG = {
    "host": "localhost",
    "port": 3306,
    "user": "retainiq_user",
    "password": getpass("Enter MySQL password for retainiq_user: "),
    "database": "retainiq",
}

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)


def execute_many(sql, rows):
    connection = None
    cursor = None
    try:
        connection = get_connection()
        cursor = connection.cursor()
        cursor.executemany(sql, rows)
        connection.commit()
    except Error as exc:
        if connection is not None:
            connection.rollback()
        raise RuntimeError(f"MySQL write failed: {exc}") from exc
    finally:
        if cursor is not None:
            cursor.close()
        if connection is not None and connection.is_connected():
            connection.close()

print_result("MySQL publishing connection is configured and ready.")

Result: MySQL publishing connection is configured and ready.


### Result & conclusion

The publication step uses the same dedicated `retainiq_user` connection used in earlier phases. I keep the password interactive so no credential is written into the notebook or Git repository.

In [11]:
# Publish the segment priority table.
connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor()

    # Ensure Phase 7 reporting tables exist before writing
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS retention_strategy_priority (
            priority_id BIGINT NOT NULL AUTO_INCREMENT,
            scope VARCHAR(20) NOT NULL,
            entity_name VARCHAR(220) NOT NULL,
            state VARCHAR(100) NULL,
            city VARCHAR(150) NULL,
            segment_name VARCHAR(120) NULL,
            retention_profile VARCHAR(60) NULL,
            customers INT NOT NULL,
            churn_rate_pct DECIMAL(8,2) NULL,
            avg_cltv DECIMAL(14,2) NULL,
            revenue_at_risk DECIMAL(16,2) NOT NULL,
            ease_of_intervention_score DECIMAL(6,2) NULL,
            priority_score DECIMAL(18,2) NOT NULL,
            priority_basis VARCHAR(120) NOT NULL,
            action_theme VARCHAR(180) NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (priority_id),
            INDEX idx_priority_scope (scope),
            INDEX idx_priority_score (priority_score),
            INDEX idx_priority_revenue (revenue_at_risk)
        ) ENGINE=InnoDB;
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS retention_impact_scenarios (
            scenario_id BIGINT NOT NULL AUTO_INCREMENT,
            scope VARCHAR(20) NOT NULL,
            entity_name VARCHAR(220) NOT NULL,
            revenue_at_risk DECIMAL(16,2) NOT NULL,
            recovery_rate_pct DECIMAL(8,2) NOT NULL,
            estimated_recovered_revenue DECIMAL(16,2) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (scenario_id),
            INDEX idx_scenario_scope (scope),
            INDEX idx_scenario_recovery (recovery_rate_pct)
        ) ENGINE=InnoDB;
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS retention_action_framework (
            segment_id INT NOT NULL,
            segment_name VARCHAR(120) NOT NULL,
            retention_profile VARCHAR(60) NOT NULL,
            customers INT NOT NULL,
            avg_cltv DECIMAL(14,2) NULL,
            churn_rate_pct DECIMAL(8,2) NULL,
            revenue_at_risk DECIMAL(16,2) NOT NULL,
            priority_score DECIMAL(18,2) NOT NULL,
            action_theme VARCHAR(180) NOT NULL,
            action_description TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (segment_id)
        ) ENGINE=InnoDB;
    """)
    connection.commit()

    cursor.execute("DELETE FROM retention_strategy_priority WHERE scope = 'segment'")
    rows = []
    for row in segment_priority.itertuples(index=False):
        rows.append((
            "segment", row.segment_name, None, None, row.segment_name,
            row.retention_profile, int(row.customers), float(row.churn_rate_pct),
            float(row.avg_cltv), float(row.revenue_at_risk),
            int(row.ease_of_intervention_score), float(row.priority_score),
            row.priority_basis,
            action_framework.loc[action_framework["segment_name"] == row.segment_name, "action_theme"].iloc[0]
        ))
    cursor.executemany("""
        INSERT INTO retention_strategy_priority
        (scope, entity_name, state, city, segment_name, retention_profile, customers,
         churn_rate_pct, avg_cltv, revenue_at_risk, ease_of_intervention_score,
         priority_score, priority_basis, action_theme)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, rows)
    connection.commit()
    print_result(f"Published {len(rows)} segment priority rows to MySQL.")
finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()


Result: Published 3 segment priority rows to MySQL.


### Result & conclusion

The segment strategy table is now persisted in MySQL with the business score, financial exposure, retention profile, and action theme stored together.

In [12]:
# Publish state and city priorities.
connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor()
    cursor.execute("DELETE FROM retention_strategy_priority WHERE scope IN ('state','city')")

    city_rows = []
    for row in city_priority.itertuples(index=False):
        city_rows.append((
            "city", f"{row.state} — {row.city}", row.state, row.city, None,
            None, int(row.customers), float(row.churn_rate_pct), float(row.avg_cltv),
            float(row.revenue_at_risk),
            int(row.ease_of_intervention_score) if pd.notna(row.ease_of_intervention_score) else None,
            float(row.priority_score), row.priority_basis, None
        ))

    state_rows = []
    for row in state_priority.itertuples(index=False):
        state_rows.append((
            "state", row.state, row.state, None, None,
            None, int(row.customers), float(row.churn_rate_pct), float(row.avg_cltv),
            float(row.revenue_at_risk),
            float(row.ease_of_intervention_score) if pd.notna(row.ease_of_intervention_score) else None,
            float(row.priority_score), row.priority_basis, None
        ))

    insert_sql = """
        INSERT INTO retention_strategy_priority
        (scope, entity_name, state, city, segment_name, retention_profile, customers,
         churn_rate_pct, avg_cltv, revenue_at_risk, ease_of_intervention_score,
         priority_score, priority_basis, action_theme)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """
    cursor.executemany(insert_sql, city_rows + state_rows)
    connection.commit()
    print_result(f"Published {len(city_rows):,} city rows and {len(state_rows):,} state rows to MySQL.")
finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()

Result: Published 1,106 city rows and 1 state rows to MySQL.


### Result & conclusion

The same reporting table now holds segment, city, and state priorities while preserving the scope. Geography rows retain a geography-specific score only when one exists; otherwise their priority is explicitly marked as revenue-at-risk-only.

In [13]:
# Publish recovery scenarios and action framework.
connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor()

    # Ensure tables exist
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS retention_impact_scenarios (
            scenario_id BIGINT NOT NULL AUTO_INCREMENT,
            scope VARCHAR(20) NOT NULL,
            entity_name VARCHAR(220) NOT NULL,
            revenue_at_risk DECIMAL(16,2) NOT NULL,
            recovery_rate_pct DECIMAL(8,2) NOT NULL,
            estimated_recovered_revenue DECIMAL(16,2) NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (scenario_id),
            INDEX idx_scenario_scope (scope),
            INDEX idx_scenario_recovery (recovery_rate_pct)
        ) ENGINE=InnoDB;
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS retention_action_framework (
            segment_id INT NOT NULL,
            segment_name VARCHAR(120) NOT NULL,
            retention_profile VARCHAR(60) NOT NULL,
            customers INT NOT NULL,
            avg_cltv DECIMAL(14,2) NULL,
            churn_rate_pct DECIMAL(8,2) NULL,
            revenue_at_risk DECIMAL(16,2) NOT NULL,
            priority_score DECIMAL(18,2) NOT NULL,
            action_theme VARCHAR(180) NOT NULL,
            action_description TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (segment_id)
        ) ENGINE=InnoDB;
    """)
    connection.commit()

    cursor.execute("DELETE FROM retention_impact_scenarios")
    cursor.execute("DELETE FROM retention_action_framework")
    scenario_rows = [
        (
            row.scope, row.entity_name, float(row.revenue_at_risk),
            float(row.recovery_rate_pct), float(row.estimated_recovered_revenue)
        )
        for row in impact_scenarios.itertuples(index=False)
    ]
    action_rows = [
        (
            int(row.segment_id), row.segment_name, row.retention_profile,
            int(row.customers), float(row.avg_cltv), float(row.churn_rate_pct),
            float(row.revenue_at_risk), float(row.priority_score),
            row.action_theme, row.action_description
        )
        for row in action_framework.itertuples(index=False)
    ]

    cursor.executemany("""
        INSERT INTO retention_impact_scenarios
        (scope, entity_name, revenue_at_risk, recovery_rate_pct, estimated_recovered_revenue)
        VALUES (%s,%s,%s,%s,%s)
    """, scenario_rows)

    cursor.executemany("""
        INSERT INTO retention_action_framework
        (segment_id, segment_name, retention_profile, customers, avg_cltv,
         churn_rate_pct, revenue_at_risk, priority_score, action_theme, action_description)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, action_rows)

    connection.commit()
    print_result(f"Published {len(scenario_rows):,} scenario rows and {len(action_rows)} action-framework rows to MySQL.")
finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()

Result: Published 4,436 scenario rows and 3 action-framework rows to MySQL.


### Result & conclusion

The scenario model and action framework are now available in MySQL for Power BI, SQL analysis, and later GenAI/RAG work.

In [14]:
# Final row-count validation and view publishing.
connection = None
cursor = None
try:
    connection = get_connection()
    cursor = connection.cursor(dictionary=True)

    # Create or replace reporting views
    cursor.execute("""
        CREATE OR REPLACE VIEW vw_retention_strategy_priority AS
        SELECT
            priority_id, scope, entity_name, state, city, segment_name,
            retention_profile, customers, churn_rate_pct, avg_cltv,
            revenue_at_risk, ease_of_intervention_score, priority_score,
            priority_basis, action_theme, created_at
        FROM retention_strategy_priority;
    """)
    cursor.execute("""
        CREATE OR REPLACE VIEW vw_retention_impact_scenarios AS
        SELECT
            scope, entity_name, revenue_at_risk, recovery_rate_pct,
            estimated_recovered_revenue, created_at
        FROM retention_impact_scenarios;
    """)
    cursor.execute("""
        CREATE OR REPLACE VIEW vw_retention_action_framework AS
        SELECT
            segment_id, segment_name, retention_profile, customers,
            avg_cltv, churn_rate_pct, revenue_at_risk, priority_score,
            action_theme, action_description, created_at
        FROM retention_action_framework;
    """)
    if connection.is_connected():
        connection.commit()

    cursor.execute("""
        SELECT
            (SELECT COUNT(*) FROM retention_strategy_priority) AS priority_rows,
            (SELECT COUNT(*) FROM retention_impact_scenarios) AS scenario_rows,
            (SELECT COUNT(*) FROM retention_action_framework) AS action_rows;
    """)
    validation = pd.DataFrame(cursor.fetchall())
finally:
    if cursor is not None:
        cursor.close()
    if connection is not None and connection.is_connected():
        connection.close()

print_result(
    f"MySQL validation returned {int(validation.loc[0, 'priority_rows']):,} priority rows, {int(validation.loc[0, 'scenario_rows']):,} scenario rows, and {int(validation.loc[0, 'action_rows']):,} action rows."
)
display(validation)


Result: MySQL validation returned 1,110 priority rows, 4,436 scenario rows, and 3 action rows.


,priority_rows,scenario_rows,action_rows
0,1110,4436,3


### Result & conclusion

The final validation confirms that the Phase 7 business-intelligence tables contain published rows after the write operations. The exact counts will depend on the number of segments, states, cities, and scenario combinations in the user's executed Phase 7 run.

In [15]:
# Final portable exports.
segment_priority.to_csv(OUTPUT_DIR / "segment_priority.csv", index=False)
state_priority.to_csv(OUTPUT_DIR / "state_priority.csv", index=False)
city_priority.to_csv(OUTPUT_DIR / "city_priority.csv", index=False)
action_framework.to_csv(OUTPUT_DIR / "retention_action_framework.csv", index=False)
impact_scenarios.to_csv(OUTPUT_DIR / "retention_impact_scenarios.csv", index=False)

print_result("Phase 7 portable CSV outputs have been refreshed after the final publication step.")

Result: Phase 7 portable CSV outputs have been refreshed after the final publication step.


### Result & conclusion

Phase 7 is now packaged as both a business-facing output layer and a database-ready reporting layer. The memo, structured summary, CSVs, tables, and views form the hand-off into Phase 8 benchmarking and Phase 9 GenAI synthesis.